## Mini Project 2!!!

In [22]:
import pandas as pd                          
import matplotlib.pyplot as plt 
import numpy as np
import cmocean 
import math
import xarray as xr
import zarr  # to read in Zarr format
import gcsfs # to read google cloud data
import momlevel as ml
import cftime

%matplotlib inline                           
%config InlineBackend.figure_format='retina' 
plt.rcParams['figure.figsize'] = 12,6 

In [23]:
ees_climate_dir = 'C:/Users/TU_Climate24_Student/Desktop/ees_climate/' 

In [25]:
seaice_observations = xr.open_zarr(ees_climate_dir+'Unit3-Pszczolkowski/data/seaice_concentration_data')
seaice_gridcell_area = xr.open_dataset(ees_climate_dir+'Unit3-Pszczolkowski/data/cell_area_seaiceconc.nc')
SST_OISST = xr.open_dataset(ees_climate_dir+'Unit3-Pszczolkowski/data/sst.oisst.mon.mean.1993_2022.nc', use_cftime=True)
OISST_landsea_mask = xr.open_dataset(ees_climate_dir+'Unit3-Pszczolkowski/data/lsmask.nc')

In [ ]:
seaice_concentration_data = (ees_climate+'Unit3-Pszczolkowski/data/seaice_concentration_data')
cell_area_seaiceconc.nc  seaice_concentration_data       sst.oisst.mon.mean.1993_2022.nc
lsmask.nc                sst.oisst.mon.ltm.1991-2020.nc

In [16]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')

In [17]:
gfdl_esm4_sea_ice_data = df.query(
    "activity_id=='ScenarioMIP' & \
    table_id == 'SImon' & \
    variable_id == 'siconc' & \
    experiment_id == 'ssp585' & \
    institution_id == 'NOAA-GFDL' &  \
    source_id == 'GFDL-ESM4' & \
    grid_label == 'gn' & \
    member_id == 'r1i1p1f1'"
)

# this only needs to be created once
gcs = gcsfs.GCSFileSystem(token='anon')
# get the path to a specific zarr store (the first one from the dataframe above)
zstore_siconc = gfdl_esm4_sea_ice_data.zstore.values[-1]
# create a mutable-mapping-style interface to the store
mapper_siconc = gcs.get_mapper(zstore_siconc)
# open the dataset using xarray and zarr
gfdl_esm4_historical_siconc = xr.open_zarr(mapper_siconc, 
                                         consolidated=True,use_cftime=True
                                         ).siconc.rename({'y':'latitude',
                                                             'x':'longitude',
                                                             'lat':'geolat',
                                                             'lon':'geolon'})
gfdl_esm4_historical_siconc

<xarray.DataArray 'siconc' (time: 1032, latitude: 576, longitude: 720)>
dask.array<open_dataset-siconc, shape=(1032, 576, 720), dtype=float32, chunksize=(600, 576, 720), chunktype=numpy.ndarray>
Coordinates:
    geolat     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
    geolon     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
  * time       (time) object 2015-01-16 12:00:00 ... 2100-12-16 12:00:00
  * longitude  (longitude) float64 -299.8 -299.2 -298.8 ... 58.75 59.25 59.75
  * latitude   (latitude) float64 -77.91 -77.72 -77.54 ... 89.47 89.68 89.89
Attributes:
    cell_measures:  area: areacello
    cell_methods:   area: mean where sea time: mean
    long_name:      Sea Ice Area Fraction (Ocean Grid)
    original_name:  siconc
    standard_name:  sea_ice_area_fraction
    units:          %

In [18]:
gfdl_esm4_cell_area_data = df.query(
    "activity_id=='CMIP' & \
    table_id == 'Ofx' & \
    variable_id == 'areacello' & \
    experiment_id == 'piControl' & \
    institution_id == 'NOAA-GFDL' &  \
    source_id == 'GFDL-ESM4' & \
    grid_label == 'gn'")

# get the path to a specific zarr store (the first one from the dataframe above)
zstore_area = gfdl_esm4_cell_area_data.zstore.values[-1]
# create a mutable-mapping-style interface to the store
mapper_area = gcs.get_mapper(zstore_area)
# open the dataset using xarray and zarr
gfdl_esm4_cell_area = xr.open_zarr(mapper_area, 
                                         consolidated=True,
                                         ).areacello.rename({'y':'latitude',
                                                             'x':'longitude',
                                                             'lat':'geolat',
                                                             'lon':'geolon'})
gfdl_esm4_cell_area

<xarray.DataArray 'areacello' (latitude: 576, longitude: 720)>
dask.array<open_dataset-areacello, shape=(576, 720), dtype=float32, chunksize=(576, 720), chunktype=numpy.ndarray>
Coordinates:
    geolat     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
    geolon     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
  * longitude  (longitude) float64 -299.8 -299.2 -298.8 ... 58.75 59.25 59.75
  * latitude   (latitude) float64 -77.91 -77.72 -77.54 ... 89.47 89.68 89.89
Attributes:
    cell_methods:   area: sum
    long_name:      Grid-Cell Area
    original_name:  areacello
    standard_name:  cell_area
    units:          m2

In [19]:
gfdl_esm4_sst_data = df.query(
    "activity_id=='ScenarioMIP' & \
    table_id == 'Omon' & \
    variable_id == 'tos' & \
    experiment_id == 'ssp585' & \
    institution_id == 'NOAA-GFDL' &  \
    source_id == 'GFDL-ESM4' & \
    grid_label == 'gn' & \
    member_id == 'r1i1p1f1'"
)

# this only needs to be created once
gcs = gcsfs.GCSFileSystem(token='anon')
# get the path to a specific zarr store (the first one from the dataframe above)
zstore_sst = gfdl_esm4_sst_data.zstore.values[-1]
# create a mutable-mapping-style interface to the store
mapper_sst = gcs.get_mapper(zstore_sst)
# open the dataset using xarray and zarr
gfdl_esm4_historical_sst = xr.open_zarr(mapper_sst, 
                                         consolidated=True,use_cftime=True
                                         ).tos.rename({'y':'latitude',
                                                             'x':'longitude',
                                                             'lat':'geolat',
                                                             'lon':'geolon'})
gfdl_esm4_historical_sst

<xarray.DataArray 'tos' (time: 1032, latitude: 576, longitude: 720)>
dask.array<open_dataset-tos, shape=(1032, 576, 720), dtype=float32, chunksize=(64, 576, 720), chunktype=numpy.ndarray>
Coordinates:
    geolat     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
    geolon     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
  * time       (time) object 2015-01-16 12:00:00 ... 2100-12-16 12:00:00
  * longitude  (longitude) float64 -299.8 -299.2 -298.8 ... 58.75 59.25 59.75
  * latitude   (latitude) float64 -77.91 -77.72 -77.54 ... 89.47 89.68 89.89
Attributes:
    cell_measures:  area: areacello
    cell_methods:   area: mean where sea time: mean
    long_name:      Sea Surface Temperature
    original_name:  tos
    standard_name:  sea_surface_temperature
    units:          degC

In [20]:
gfdl_esm4_cell_area_data = df.query(
    "activity_id=='CMIP' & \
    table_id == 'Ofx' & \
    variable_id == 'areacello' & \
    experiment_id == 'piControl' & \
    institution_id == 'NOAA-GFDL' &  \
    source_id == 'GFDL-ESM4' & \
    grid_label == 'gn'")

# get the path to a specific zarr store (the first one from the dataframe above)
zstore_area = gfdl_esm4_cell_area_data.zstore.values[-1]
# create a mutable-mapping-style interface to the store
mapper_area = gcs.get_mapper(zstore_area)
# open the dataset using xarray and zarr
gfdl_esm4_cell_area = xr.open_zarr(mapper_area, 
                                         consolidated=True,
                                         ).areacello.rename({'y':'latitude',
                                                             'x':'longitude',
                                                             'lat':'geolat',
                                                             'lon':'geolon'})
gfdl_esm4_cell_area

<xarray.DataArray 'areacello' (latitude: 576, longitude: 720)>
dask.array<open_dataset-areacello, shape=(576, 720), dtype=float32, chunksize=(576, 720), chunktype=numpy.ndarray>
Coordinates:
    geolat     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
    geolon     (latitude, longitude) float32 dask.array<chunksize=(576, 720), meta=np.ndarray>
  * longitude  (longitude) float64 -299.8 -299.2 -298.8 ... 58.75 59.25 59.75
  * latitude   (latitude) float64 -77.91 -77.72 -77.54 ... 89.47 89.68 89.89
Attributes:
    cell_methods:   area: sum
    long_name:      Grid-Cell Area
    original_name:  areacello
    standard_name:  cell_area
    units:          m2

In [21]:
seaice_concentration_data
cell_area_seaiceconc.nc
sst.oisst.mon.mean.1993_2022.nc 
lsmask.nc
gfdl_esm4_historical_sst
gfdl_esm4_cell_area
gfdl_esm4_historical_sst
gfdl_esm4_cell_area

SyntaxError: invalid syntax (2980122774.py, line 3)

## Required visualizations are described in the provided template notebooks:
• SeaIce_ClimateModel_Analysis_Notebook.ipynb
• SeaIce_Observations_Notebook.ipynb
• SST_ClimateModel_Analysis_Notebook.ipynb
• SST_Observations_Notebook.ipynb

### 1.) What is sea ice?? 
<span style="color:red"> Sea ice is a thin and fragile layer of frozen ocean water that forms in the Arctic/Antarctic oceans. Of course, this is not to be confused with icebergs, which are made of fresh water from compacted snow. Sea ice forms on the ocean's surface in frozen temperatures. It starts as a thin slate of crystals that grow outward into a sheet of salty ice. The sea ice then grows downward and thickens. Sea ice is dynamic, as winds and ocean currents frequently move it across the ocean's surface. The salty sea ice though is possibly the most dangerous symptom of the Earth's rising temperatures.n.


### 2.) What role does sea ice play in our climate system and why should we be concerned about its rapid decline?
 <span style="color:red"> Sea ice plays a pivotal role in our climate system for several key reasons. First, it helps regulate temperatures by reflecting sunlight, which keeps the polar regions cool. When the sea ice melts, it exposes darker ocean water, which then absorbs more heat and contributes to further warming of our planet.

<span style="color:red"> Additionally, sea ice is essential for various wildlife species, providing habitats for polar bears, seals, and penguins, while also supporting complex marine ecosystems. The influence it has on ocean currents and circulation patterns, which are crucial for distributing heat around the planet.

<span style="color:red"> The rapid decline of sea ice raises significant concerns. It disrupts ecosystems and threatens the survival of many species. Furthermore, changes in sea ice can alter weather patterns and climate systems on a global scale. Although melting sea ice itself doesn’t directly raise sea levels, it can destabilize land ice, leading to increased sea levels.


### 3.) Define the following terms --- be sure to also define their units.
##### <span style="color:red"> Sea Ice Concentration (%)

 <span style="color:red"> Sea ice concentration measures the proportion of a specific area of ocean that is covered by sea ice. It is expressed as a percentage, indicating how much of the ocean surface is occupied by ice relative to the total area.

 ##### <span style="color:red"> Sea Ice Extent  (km² & mi²)

 <span style="color:red"> Sea ice extent refers to the total area of ocean where the concentration of sea ice is at least 15%. This measurement provides insight into the overall coverage of sea ice in a given area.

 
 ##### <span style="color:red"> Sea Ice Thickness  (m & cm)

 <span style="color:red"> Sea ice thickness indicates the vertical measurement of the ice layer, from the surface down to its base. This metric is crucial for understanding the volume of ice and its implications for the climate and ecosystems.


### 4.) How is Antarctic sea ice different from Arctic sea ice and why? Your answer should discuss its seasonal cycle, thickness, and extent.

#### <span style="color:red"> Seasonal Cycle

 ##### <span style="color:red"> Arctic Sea Ice: 
 
 <span style="color:red"> In the Arctic, sea ice has a distinct seasonal pattern. It forms during the winter and typically reaches its largest extent around March. As summer approaches, the ice melts significantly, with the lowest extent usually seen in September. This melting is driven by warmer temperatures and increased sunlight during the summer months.

 ##### <span style="color:red"> Antarctic Sea Ice: 

 <span style="color:red"> In contrast, Antarctic sea ice also grows and melts with the seasons, but the cycle is less extreme. It reaches its maximum extent in September as well, but the melt during the summer (around February) tends to leave more ice remaining than in the Arctic. This is partly because the Antarctic has a more stable environment, with colder temperatures persisting longer.

#### <span style="color:red"> Thickness
 
 ##### <span style="color:red"> Arctic Sea Ice: 

 <span style="color:red"> The ice in the Arctic is generally thicker than that in the Antarctic. This is due to the presence of multiyear ice—ice that survives several melting seasons—which accumulates over time, leading to greater thickness.

##### <span style="color:red"> Antarctic Sea Ice: 

 <span style="color:red"> Antarctic sea ice is typically thinner and more variable. Much of it is seasonal, forming and melting within a single year. This results in less overall thickness compared to the Arctic.

####  <span style="color:red"> Extent
##### <span style="color:red"> Arctic Sea Ice: 

 <span style="color:red"> The extent of Arctic sea ice has been declining rapidly over the last few decades, reflecting the impacts of climate change. This decline is evident in both the maximum and minimum extents.

##### <span style="color:red"> Antarctic Sea Ice:

 <span style="color:red"> Interestingly, Antarctic sea ice extent has shown more fluctuations and, at times, even slight increases in recent years. This variability is influenced by regional factors, such as ocean currents and wind patterns.

### 5.) Is multi-year ice more common in the Arctic or Antarctic?
<span style="color:red"> Multi-year ice is more common in the Arctic compared to the Antarctic. In the Arctic, multi-year ice forms when first-year sea ice survives the summer melt season, allowing it to thicken over several years. On the contrary, the Antarctic experiences more seasonal melting and less persistent sea ice.

(NSIDC) nsidc.org.

### 6.) What month has experienced the largest change in sea ice extent when comparing 2020 to the 1982 to 2010 average for the Arctic and Antarctic? 

### 7.) Describe the term Arctic amplification and the feedback that is believed to be the key driver of sea ice loss in the Arctic.

### 8.) Does sea ice melt contribute to global sea level rise? Why or why not?

### 9.) How does the projected rate of change in SST compare when considering the areaweighted global average, Arctic average, and Antarctic average?

### 10.) Provide a well communicated explanation of what the SSP5-8.5 scenario is (https://www.carbonbrief.org/explainer-how-shared-socioeconomic-pathways-explorefuture-climate-change/). Does this scenario align with the Paris Climate goal of keeping warming below 1.5 C?

### 11.) After reading through a brief introduction to the SSPs via the link above, what path do you think we are on / will likely end up on? Why?